<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/35_mlp_pytorch.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Red feedforward con PyTorch

**Pregunta guía:** ¿Cómo controlamos explícitamente el ciclo de entrenamiento?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere:** runtime estándar de Colab con PyTorch. Usaremos el mismo
tipo de problema que en TensorFlow, pero escribiremos `Dataset`, lotes,
forward, backward, actualización, validación y early stopping.


In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEMILLA = 42
torch.manual_seed(SEMILLA)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X, y = make_moons(n_samples=1_600, noise=0.24, random_state=SEMILLA)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=SEMILLA)
scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = map(scaler.transform, [X_train, X_val, X_test])

def tensor_dataset(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y[:,None], dtype=torch.float32))

generador = torch.Generator().manual_seed(SEMILLA)
train_loader = DataLoader(tensor_dataset(X_train, y_train), batch_size=64, shuffle=True, generator=generador)
val_loader = DataLoader(tensor_dataset(X_val, y_val), batch_size=256)
print("PyTorch", torch.__version__, "dispositivo", dispositivo)


In [ ]:
class MLP(nn.Module):
    def __init__(self, unidades=32):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(2, unidades), nn.ReLU(),
            nn.Linear(unidades, unidades), nn.ReLU(),
            nn.Linear(unidades, 1),
        )
    def forward(self, x):
        return self.red(x)

modelo = MLP().to(dispositivo)
print(modelo)
print("parámetros:", sum(p.numel() for p in modelo.parameters() if p.requires_grad))


In [ ]:
def pérdida_media(modelo, loader, criterio):
    modelo.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            total += criterio(modelo(xb), yb).item() * len(xb)
            n += len(xb)
    return total / n

criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
mejor_estado, mejor_val, paciencia = None, np.inf, 0
historia = {"train": [], "val": []}

for época in range(150):
    modelo.train()
    acumulada = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(dispositivo), yb.to(dispositivo)
        optimizador.zero_grad()
        logits = modelo(xb)
        pérdida = criterio(logits, yb)
        pérdida.backward()
        optimizador.step()
        acumulada += pérdida.item() * len(xb)
    train_loss = acumulada / len(train_loader.dataset)
    val_loss = pérdida_media(modelo, val_loader, criterio)
    historia["train"].append(train_loss); historia["val"].append(val_loss)
    if val_loss < mejor_val - 1e-5:
        mejor_val = val_loss; mejor_estado = copy.deepcopy(modelo.state_dict()); paciencia = 0
    else:
        paciencia += 1
    if paciencia >= 15: break

modelo.load_state_dict(mejor_estado)
print("épocas:", len(historia["train"]), "mejor val:", mejor_val)


In [ ]:
modelo.eval()
with torch.no_grad():
    logits = modelo(torch.tensor(X_test, dtype=torch.float32, device=dispositivo))
    prob = torch.sigmoid(logits).cpu().numpy().ravel()
print("accuracy test:", accuracy_score(y_test, prob >= 0.5))
print("log loss test:", log_loss(y_test, prob))
plt.plot(historia["train"], label="train")
plt.plot(historia["val"], label="validation")
plt.xlabel("época"); plt.ylabel("BCE"); plt.legend(); plt.show()


**Lectura del ciclo:** `zero_grad` evita acumular gradientes; `backward`
aplica diferenciación automática; `step` actualiza; `eval` cambia el
comportamiento de dropout/batch normalization; `no_grad` evita construir
un grafo en inferencia.

**Ejercicios:** omita deliberadamente `zero_grad`; añada dropout; registre
normas de gradiente; compare con TensorFlow usando misma arquitectura,
partición, épocas máximas y criterio de selección.
